# PennyLane Apple Silicon scaling

Sweep statevector widths around MettleQ's automatic GPU crossover and verify complete-state parity.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    total_variation_distance,
)

In [2]:
import statistics

def make_qnode(device, width):
    @qml.qnode(device)
    def circuit():
        for wire in range(width):
            qml.RY(0.03 * (wire + 1), wires=wire)
        for wire in range(width - 1):
            qml.CNOT(wires=[wire, wire + 1])
        return qml.state()
    return circuit

rows = []
for width in (12, 14, 16):
    reference_qnode = make_qnode(qml.device("default.qubit", wires=width), width)
    reference, reference_ms, _ = benchmark(reference_qnode, repeats=2)
    mettleq_device = MettleQDevice(wires=width, method="statevector", device="auto")
    mettleq_qnode = make_qnode(mettleq_device, width)
    candidate, mettleq_ms, _ = benchmark(mettleq_qnode, repeats=2)
    method, device = pennylane_selection(mettleq_device)
    rows.append({
        "width": width,
        "reference_ms": reference_ms,
        "mettleq_ms": mettleq_ms,
        "error": phase_aligned_statevector_error(reference, candidate),
        "method": method,
        "device": device,
    })

tutorial_result = emit_result(
    notebook="pennylane/13_apple_gpu_scaling.ipynb",
    framework="pennylane",
    reference_ms=statistics.median(row["reference_ms"] for row in rows),
    mettleq_ms=statistics.median(row["mettleq_ms"] for row in rows),
    check="per-width statevector atol=3e-6 and policy-selected GPU",
    passed=all(row["error"] <= 3e-6 for row in rows) and rows[-1]["device"] == "gpu",
    exact_match=all(row["error"] == 0.0 for row in rows),
    selected_method=rows[-1]["method"],
    selected_device=rows[-1]["device"],
    metrics={"widths": rows},
    notes="The aggregate medians summarize different widths; use per-width timings for interpretation.",
)

TUTORIAL_RESULT::{"check": "per-width statevector atol=3e-6 and policy-selected GPU", "exact_match": false, "framework": "pennylane", "machine": "arm64", "metrics": {"widths": [{"device": "cpu", "error": 1.0412182105401513e-07, "method": "statevector", "mettleq_ms": 1.6983749956125394, "reference_ms": 1.3478539913194254, "width": 12}, {"device": "gpu", "error": 1.3858108138808944e-07, "method": "statevector", "mettleq_ms": 2.9038954962743446, "reference_ms": 1.8154374993173406, "width": 14}, {"device": "gpu", "error": 9.131542633156187e-08, "method": "statevector", "mettleq_ms": 4.002208501333371, "reference_ms": 4.051854513818398, "width": 16}]}, "mettleq_median_ms": 2.9038954962743446, "notebook": "pennylane/13_apple_gpu_scaling.ipynb", "notes": "The aggregate medians summarize different widths; use per-width timings for interpretation.", "passed": true, "python": "3.13.2", "reference_median_ms": 1.8154374993173406, "reference_over_mettleq": 0.6251731515980931, "schema_version": 1, "